# Mueller-matrix rendering of a sphere

Pick a material, a camera and a light; get the per-pixel 4×4 Mueller matrix.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import spheremm as sm
from spheremm.plot import plot_intensity

list(sm.library.MATERIALS)

## Scene
Units are metres. `DirectionalLight.from_camera(camera, phase_angle, azimuth)` places a collimated beam at `phase_angle` degrees (light–sphere–camera angle; 0 = coaxial back-scattering). `azimuth` sets the side the light comes from (0° = image right, 90° = top). `PointLight.from_camera(..., distance=0.5)` does the same with a point source.

In [ ]:
camera = sm.Camera(origin=[0, 0, 0.4], resolution=200, width=0.05)  # orthographic; use fov=... for a pinhole
sphere = sm.Sphere(radius=0.02)
light = sm.DirectionalLight.from_camera(camera, phase_angle=45)

## Library material
`sm.library.get(name, **overrides)`; e.g. change the roughness or the diffuse albedo of your samples.

In [ ]:
img = sm.render_mueller(sm.library.get("brass"), camera, light, sphere, wavelengths=[450, 550, 650])
img.M.shape  # (wavelength, H, W, 4, 4)

In [ ]:
img.plot(1)
plot_intensity(img[1]);

## Phase-angle sweep

In [ ]:
sweep = sm.render_phase_sweep(sm.library.get("red_abs"), [0, 30, 60, 90], camera, sphere, wavelengths=650)
for img in sweep:
    img.plot(0, title=f"red ABS, phase {img.meta['phase_angle']:.0f}°, 650 nm")

## Polishing traces
Metals are anisotropic: `roughness=(along, across)` concentric traces centred in the image. Use a source with finite angular size, otherwise only M00 changes.

In [ ]:
brass = sm.library.get("brass", roughness=(0.03, 0.3))
wide = sm.DirectionalLight.from_camera(camera, phase_angle=30, angular_radius=5, samples=128)
img = sm.render_mueller(brass, camera, wide, sphere)
img.plot()
plot_intensity(img[0]);

## Custom material
* `sm.Conductor(ior, roughness)`: metals, `ior` = name (`sm.available_iors()`) or complex `n + 1j*k`
* `sm.Dielectric(ior, roughness, albedo)`: plastics/ceramics (polarizing specular + diffuse); `albedo` may be a function of wavelength
* `sm.Lambertian(albedo)`: ideal depolarizer
* `ior` may also be a function of wavelength (nm) or `sm.ior_from_csv("file.csv")` from refractiveindex.info

In [ ]:
custom = sm.Dielectric(ior=1.5, roughness=0.3, albedo=0.6)
sm.render_mueller(custom, camera, light, sphere).plot();

## Using the result
`img.M` is a plain NumPy array, e.g. simulate a measurement with your own PSG/PSA states:

In [ ]:
img = sweep[2]
S_in = np.array([1, 1, 0, 0])            # horizontal linear from the PSG
S_out = img[0] @ S_in                     # (H, W, 4) Stokes image
dolp, aolp = sm.polarization.stokes_to_dolp_aolp(S_out)
plt.imshow(dolp, vmin=0, vmax=1, cmap='gray'); plt.colorbar(label='DoLP'); plt.axis('off');